> 在线讲义：[第 31 章 · 从单卡到多卡：放得下、算得快、接得多](https://wikieden.github.io/ai-infra-from-constraints/lessons/31-multi-gpu.html) · 本书《从约束推导 AI Infra》

# 第 31 章 · 从单卡到多卡：放得下、算得快、接得多

配套讲义：仓库里 `ai-infra/notes/lessons/31-multi-gpu.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



HGX H100 八卡与 Qwen3-235B-A22B 的容量、专家读取时间取自 AII 第 6 章 §6.1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
KiB, MiB, GiB = 2**10, 2**20, 2**30
HBM, PEAK = 3350e9, 989.4e12  # H100 SXM

**函数 `kv_per_token`**：每 token 的 KV 字节：K、V 各一份 × 层 × KV 头 × 头维 × 字节。

In [ ]:
def kv_per_token(layers: int, kv_heads: int, head_dim: int, b: int = 2) -> int:
    """每 token 的 KV 字节：K、V 各一份 × 层 × KV 头 × 头维 × 字节。"""
    return 2 * layers * kv_heads * head_dim * b

**函数 `per_card_gb`**：每卡内存：权重分片 + KV 份额 + 每卡一份工作区。

In [ ]:
def per_card_gb(weights_gb: float, kv_gb: float, cards: int, kv_share: float, work_gb: float = 2 * GiB / 1e9,
                weights_per_card_gb: float | None = None) -> float:
    """每卡内存：权重分片 + KV 份额 + 每卡一份工作区。"""
    w = weights_per_card_gb if weights_per_card_gb is not None else weights_gb / cards
    return w + kv_gb * kv_share + work_gb

**函数 `op_time`**：式 (6-2)：(计算 s, 读取 s, max)。

In [ ]:
def op_time(flops: float, nbytes: float, peak: float = PEAK, bw: float = HBM) -> tuple[float, float, float]:
    """式 (6-2)：(计算 s, 读取 s, max)。"""
    tc, tm = flops / peak, nbytes / bw
    return tc, tm, max(tc, tm)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：Qwen3-235B-A22B，8192 个上下文 token")
kv = kv_per_token(94, 4, 128)
kv_gb = kv * 8192 / 1e9
print(f"  每 token KV {kv // KiB} KiB，8192 个 {kv_gb:.2f} GB；单卡共需 {470.2 + kv_gb + 2 * GiB / 1e9:.1f} GB")
pc = per_card_gb(470.2, kv_gb, 8, 1 / 4, weights_per_card_gb=58.96)
print(f"  八卡每卡 {pc:.2f} GB（KV 每卡存 1/4：{kv_gb / 4:.2f} GB）；空余可再放 {int((80 - 58.96 - 2 * GiB / 1e9) / (kv_gb / 4))} 个会话")

In [ ]:
print("\n第二步：一层专家，单 token 选 8 个")
w = 8 * 3 * 4096 * 1536 * 2
tc, tm, _ = op_time(2 * 8 * 3 * 4096 * 1536, w)
print(f"  读 {w // MiB} MiB {tm * 1e6:.1f} μs，算 {tc * 1e6:.2f} μs，MFU 上限 {tc / tm:.1%}")
print(f"  全模型一步：单卡 {(43.1e9 + kv_gb * 1e9) / HBM * 1e3:.1f} ms；八卡每卡 {43.1 / 8 + kv_gb / 4:.2f} GB，{(43.1e9 / 8 + kv_gb * 1e9 / 4) / HBM * 1e3:.2f} ms")
tc, tm, _ = op_time(2 * 8192 * 8 * 3 * 4096 * 1536, 128 * 3 * 4096 * 1536 * 2)
print(f"  prefill 8192 token：读 4.5 GiB {tm * 1e3:.2f} ms，算 {tc * 1e3:.2f} ms → 计算主导")
print(f"  隐藏张量：单 token {4096 * 2 // KiB} KiB，8192 token {8192 * 4096 * 2 // MiB} MiB")

## 3. 代码里才看得到的

1. 每卡权重 58.96 GB 是原文给的精确值（含每卡一份的归一化和路由器），代码直接用它，不是 470.2/8。
2. KV 每卡存 1/4 而不是 1/8：4 个 KV 头分给 8 张卡，每个头存两份。
3. 2 GiB 工作区换成十进制是 2.147 GB，原文表里写 2.15。

## 4. 自己改一改

- 把 `op_time` 换成 H200（4.8 TB/s），单 token 一层专家省多少？
- 让 `per_card_gb` 接受 KV 头数和卡数，自动算每卡 KV 份额。
- 算 Qwen3-235B 用 FP8 权重时八卡每卡还能多放几个 8K 会话。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 装不下、跑得慢、接不过来，三种病三种药；
> 一套书分八个架，目录每架一本；
> 送快递花在路上，拉满车花在装卸；
> 套娃从外到里：请求、层、专家、特征、位置。